## Get Size of s3 objects

Let us go through the details about how we can get size of s3 objects using `MaxKeys` and `Marker`. We will improvise on top of getting count of s3 objects.

* Here is the code used to get count of objects in s3.

```python
marker = ''
object_count = 0
while True:
    s3_objects = s3_client.list_objects(
        Bucket='itv-genlogs',
        Prefix='logs/year',
        Marker=marker,
        MaxKeys=200
    ).get('Contents')
    if not s3_objects:
        break
    object_count += len(s3_objects)
    marker = s3_objects[-1]['Key']
    print(marker)
```

* Create client with appropriate profile.
* Invoke `list_objects` in pages using `MaxKeys` and `Marker`.
* Each entry in the output of `list_objects` contain `Size` along with `Key` and other details.
* Add the Size of each entry to get the total size of our s3 Bucket. The size in each entry will be in Bytes and you might have to convert to mega bytes.

In [1]:
import boto3
import os
os.environ.setdefault('http_proxy', 'http://webproxy...com:')
os.environ.setdefault('https_proxy', 'http://webproxy...com:')
os.environ.setdefault('AWS_PROFILE', 'itvgenlogs')
BUCKET_NAME = 'itv-genlogs'
PREFIX = 'logs/year'

In [2]:
s3_client = boto3.client('s3')

In [7]:
marker = ''                                # initialize to empty marker
object_count = 0
MAX_KEYS = 50
objects_total_size = 0.0                   # initialize total size to zero
while True:
    s3_objects = s3_client \
        .list_objects(                     # get objects in the bucket and prefix
            Bucket=BUCKET_NAME, 
            Prefix=PREFIX, 
            Marker=marker,
            MaxKeys=MAX_KEYS               # max keys returned for each loop
        ) \
        .get('Contents')
    if not s3_objects:
        break
    object_count += len(s3_objects)
    marker = s3_objects[-1]['Key']         # get next batch after this marker (pagination)
    for obj in s3_objects:
        objects_total_size += obj['Size']
    print(marker)

logs/year=2026/month=01/day=07/gen_logs_s3-3-2026-01-07-04-06-53-23c10054-b19e-42df-b415-6332c010b31a
logs/year=2026/month=01/day=07/gen_logs_s3-3-2026-01-07-04-59-45-454a5c1e-7f26-4f03-90bd-6c4fe06aec2a
logs/year=2026/month=01/day=07/gen_logs_s3-3-2026-01-07-05-50-35-c7ad2871-d872-4f18-b5df-d875c6350484
logs/year=2026/month=01/day=07/gen_logs_s3-3-2026-01-07-05-58-43-faf6ff3e-76a5-4936-af33-553c1dbc7370


In [8]:
objects_total_size

1961335.0

In [9]:
objects_total_size / 1000000

1.961335

In [14]:
s3_objects = s3_client.list_objects(
    Bucket=BUCKET_NAME, 
    Prefix=PREFIX
)

In [ ]:
s3_objects['Contents'][0]

In [16]:
[obj['Size'] for obj in s3_objects['Contents']][:5]

[24103, 11948, 11869, 12249, 12339]